# 02 — Pipeline RAG End-to-End com Chroma

**Bloom:** Apply | **Duração:** 90 min | **Objetivos:** M2-O1, M2-O2

Referência completa: [`labs/lab-002.md`](../labs/lab-002.md). Corpus: 3 papers seminais arXiv (`datasets/corpus/`).

**Fluxo:** INGEST PDFs → CHUNK recursive 800/100 → EMBED → INDEX (Chroma) → RETRIEVE top-5 → GENERATE com contexto.

## Setup

**Provider:** Google Gemini (default da turma — free tier).  
**Como gerar a chave:** ver `API-KEYS.pdf` (mesma pasta) ou https://aistudio.google.com/apikey

**Onde colocar a chave** (a célula abaixo detecta automaticamente):

| Ambiente | Onde |
|---|---|
| **Google Colab** | 🔑 Secrets (barra lateral) → `GEMINI_API_KEY` → ligar acesso ao notebook |
| **Jupyter local** | arquivo `.env` na raiz da disciplina com `GEMINI_API_KEY=AIza...` |
| **Fallback** | prompt `getpass` (a célula pede ao rodar, sem persistir) |

**LLM + embeddings:** ambos via Gemini (Gemini tem endpoint de embeddings nativo).  
Execute as células em ordem (`Run All` funciona end-to-end após informar a chave).

In [1]:
# Bootstrap Colab — instala versoes fixas e reinicia o runtime 1x sozinho.
# No Jupyter local (uv sync) nao faz nada. Depois do restart, rode Run all.
import importlib.metadata as md
import importlib.util
import os
import subprocess
import sys

PINS = [
    "openai==2.37.0",
    "chromadb==1.5.9",
    "langchain-core==1.4.0",
    "langchain-text-splitters==1.1.2",
    "pypdf",
    "python-dotenv",
]
_ALVO = {
    "openai": "2.37.0",
    "chromadb": "1.5.9",
    "langchain-core": "1.4.0",
    "langchain-text-splitters": "1.1.2",
}


def _precisa_instalar() -> bool:
    for _nome, _ver in _ALVO.items():
        try:
            if md.version(_nome) != _ver:
                return True
        except md.PackageNotFoundError:
            return True
    return False


if importlib.util.find_spec("google.colab") is not None and _precisa_instalar():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    print("Instalado. Reiniciando o runtime... rode 'Run all' de novo.")
    os.kill(os.getpid(), 9)  # forca restart do Colab
else:
    print("Ambiente OK.")


Ambiente OK.


In [2]:
# Baixar corpus se ainda nao existir (resiliente a Colab e Jupyter local)
import subprocess
import sys
from pathlib import Path
from urllib.request import Request, urlopen

CORPUS_DIR = Path("data/corpus")
_PAPERS = {
    "attention-is-all-you-need.pdf": "https://arxiv.org/pdf/1706.03762v7.pdf",
    "rag-knowledge-intensive-nlp.pdf": "https://arxiv.org/pdf/2005.11401v4.pdf",
    "lost-in-the-middle.pdf": "https://arxiv.org/pdf/2307.03172v3.pdf",
}

if not list(CORPUS_DIR.glob("*.pdf")):
    # 1) Script oficial disponivel? Usa-o (valida SHA256). Jupyter local roda de
    #    notebooks/ (../datasets); outros ambientes tentam ./datasets.
    _script = next(
        (
            p
            for p in (Path("datasets/download.py"), Path("../datasets/download.py"))
            if p.exists()
        ),
        None,
    )
    if _script is not None:
        subprocess.run(
            [sys.executable, str(_script), "--out", str(CORPUS_DIR)], check=True
        )
    else:
        # 2) Fallback (Colab sem a pasta datasets/): baixa os 3 PDFs direto.
        CORPUS_DIR.mkdir(parents=True, exist_ok=True)
        for _name, _url in _PAPERS.items():
            _req = Request(_url, headers={"User-Agent": "aulas-ppi-corpus/1.0"})
            (CORPUS_DIR / _name).write_bytes(urlopen(_req).read())
            print("baixado:", _name)

print(f"corpus: {len(list(CORPUS_DIR.glob('*.pdf')))} PDFs em {CORPUS_DIR}/")


baixado: attention-is-all-you-need.pdf
baixado: rag-knowledge-intensive-nlp.pdf
baixado: lost-in-the-middle.pdf
corpus: 3 PDFs em data/corpus/


In [12]:
import os
from pathlib import Path

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from pypdf import PdfReader


def _load_groq_key() -> tuple[str, str]:
    """Retorna (api_key, origem). Tenta Colab Secrets -> .env local -> getpass."""
    # 1) Google Colab — Secrets manager
    try:
        from google.colab import userdata  # type: ignore
        try:
            k = userdata.get("GROQ_API_KEY")
            if k:
                return k, "Colab Secrets"
        except Exception:
            pass
    except ImportError:
        pass

    # 2) .env local (Jupyter)
    try:
        from dotenv import find_dotenv, load_dotenv
        dp = find_dotenv(usecwd=True)
        if dp:
            load_dotenv(dp)
    except ImportError:
        pass
    k = os.getenv("GROQ_API_KEY")
    if k:
        return k, ".env local"

    # 3) Fallback interativo
    from getpass import getpass
    k = getpass("Cole sua GROQ_API_KEY (https://console.groq.com/keys): ")
    if not k:
        raise RuntimeError("GROQ_API_KEY nao fornecida — abortando.")
    return k, "prompt interativo"


api_key, key_source = _load_groq_key()
os.environ["GROQ_API_KEY"] = api_key

# Cliente Groq (compatível com API OpenAI)
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1/",
)
LLM_MODEL = "llama-3.3-70b-versatile"  # rápido e gratuito no Groq

# Embeddings via sentence-transformers (gratuito, roda local)
# Modelo multilíngue leve — suporta pt-br e inglês
embed_fn = SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

CORPUS_DIR = Path("data/corpus")
PERSIST_DIR = "data/chroma"
print(f"LLM: {LLM_MODEL} (Groq) | embeddings: paraphrase-multilingual-MiniLM-L12-v2 (local) | key: {key_source}")

LLM: llama-3.3-70b-versatile (Groq) | embeddings: paraphrase-multilingual-MiniLM-L12-v2 (local) | key: .env local


## Etapa 1 — Ingestão de PDFs





In [5]:
def ingest_pdfs(corpus_dir: Path) -> list[dict]:
    docs = []
    for pdf_path in sorted(corpus_dir.glob("*.pdf")):
        reader = PdfReader(pdf_path)
        for page_idx, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                docs.append(
                    {
                        "text": text,
                        "source": pdf_path.name,
                        "page": page_idx + 1,
                    }
                )
    return docs


docs = ingest_pdfs(CORPUS_DIR)
print(f"Paginas ingeridas: {len(docs)}")
print(f"Primeira pagina: {docs[0]['source']}:p{docs[0]['page']}")
print(f"Preview: {docs[0]['text'][:200]}...")

Paginas ingeridas: 52
Primeira pagina: attention-is-all-you-need.pdf:p1
Preview: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...


## Etapa 2 — Chunking Recursivo (800 chars, overlap 100)

**Reflexão:** por que `chunk_overlap=100`? O que aconteceria com `overlap=0` em um chunk que termina no meio de uma frase importante?

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for doc in docs:
    for i, chunk in enumerate(splitter.split_text(doc["text"])):
        chunks.append(
            {
                "id": f"{doc['source']}-p{doc['page']}-c{i}",
                "text": chunk,
                "source": doc["source"],
                "page": doc["page"],
                "chunk_idx": i,
            }
        )

print(f"Total de chunks: {len(chunks)}")
print(f"Tamanho medio: {sum(len(c['text']) for c in chunks) / len(chunks):.0f} chars")

Total de chunks: 271
Tamanho medio: 700 chars


## Etapa 3 — Embedding e Indexing em Chroma





In [7]:
import time

chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
# Recria a collection limpa (evita duplicar chunks em re-execucoes da celula)
try:
    chroma_client.delete_collection("papers")
except Exception:
    pass
collection = chroma_client.get_or_create_collection(
    name="papers",
    embedding_function=embed_fn,
)

# Indexa em lotes: respeita o limite de inputs por request e o rate limit
# do free tier do Gemini (15 RPM). Pausa curta entre lotes.
BATCH = 50
for start in range(0, len(chunks), BATCH):
    lote = chunks[start : start + BATCH]
    collection.add(
        ids=[c["id"] for c in lote],
        documents=[c["text"] for c in lote],
        metadatas=[
            {"source": c["source"], "page": c["page"], "chunk_idx": c["chunk_idx"]}
            for c in lote
        ],
    )
    print(f"indexados {min(start + BATCH, len(chunks))}/{len(chunks)} chunks")
    time.sleep(1)

print(f"Indexed: {collection.count()} chunks")


indexados 50/271 chunks
indexados 100/271 chunks
indexados 150/271 chunks
indexados 200/271 chunks
indexados 250/271 chunks
indexados 271/271 chunks
Indexed: 271 chunks


**Ponto de verificação:** `collection.count()` deve casar com `len(chunks)`.

## Etapa 4 — Retrieval (top-5)

In [8]:
def retrieve(query: str, k: int = 5) -> list[dict]:
    result = collection.query(query_texts=[query], n_results=k)
    return [
        {
            "text": result["documents"][0][i],
            "source": result["metadatas"][0][i]["source"],
            "page": result["metadatas"][0][i]["page"],
            "distance": result["distances"][0][i],
        }
        for i in range(len(result["documents"][0]))
    ]


hits = retrieve("Qual e a diferenca entre fine-tuning e RAG?", k=5)
for h in hits:
    print(f"[{h['source']}:p{h['page']}] dist={h['distance']:.3f}")
    print(f"  {h['text'][:120]}...\n")

[rag-knowledge-intensive-nlp.pdf:p2] dist=0.464
  basis (where different documents are responsible for different tokens). Like T5 [51] or BART, RAG
can be ﬁne-tuned on an...

[rag-knowledge-intensive-nlp.pdf:p9] dist=0.475
  pieces of retrieved content, as well as learning latent retrieval, and retrieving evidence documents
rather than related...

[rag-knowledge-intensive-nlp.pdf:p5] dist=0.480
  4 Results
4.1 Open-domain Question Answering
Table 1 shows results for RAG along with state-of-the-art models. On all fo...

[rag-knowledge-intensive-nlp.pdf:p6] dist=0.502
  with both models outperforming BART on Q-BLEU-1. 4 shows human evaluation results, over 452
pairs of generations from BA...

[rag-knowledge-intensive-nlp.pdf:p1] dist=0.508
  pare two RAG formulations, one which conditions on the same retrieved passages
across the whole generated sequence, and ...



## Etapa 5 — Augment + Generate (com âncora ao contexto)

In [14]:
## Etapa 5 — Augment + Generate (com âncora ao contexto)

RAG_PROMPT = """Voce e um assistente tecnico especializado em NLP e RAG.
Use o contexto abaixo para responder a pergunta de forma completa e informativa.
Sintetize as informacoes dos trechos fornecidos, mesmo que a resposta nao esteja
explicita em uma unica frase.
Sempre cite a fonte usando o formato [arquivo:pagina].
So diga "Nao encontrado no corpus" se o contexto nao tiver NENHUMA relacao com a pergunta.

CONTEXTO:
{context}

PERGUNTA: {question}

RESPOSTA:"""


def rag_answer(question: str, k: int = 5) -> dict:
    hits = retrieve(question, k=k)
    context = "\n\n---\n\n".join(f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits)
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": RAG_PROMPT.format(context=context, question=question)}
        ],
        temperature=0.0,
    )
    return {
        "answer": response.choices[0].message.content,
        "sources": [(h["source"], h["page"]) for h in hits],
    }


# Teste
QUERIES = [
    "Qual e a diferenca entre fine-tuning e RAG?",
    "O que e chunking e por que importa?",
    "Que metricas avaliam qualidade de retrieval?",
    "Como reduzir alucinacao em LLM?",
    "Qual o papel de embeddings em busca semantica?",
]

for q in QUERIES:
    out = rag_answer(q)
    print(f"\nQ: {q}")
    print(f"A: {out['answer']}")
    print(f"Fontes: {out['sources']}")


Q: Qual e a diferenca entre fine-tuning e RAG?
A: A diferença entre fine-tuning e RAG (Retrieval-Augmented Generation) está na forma como os modelos de linguagem são treinados e utilizam o conhecimento.

O fine-tuning é um processo de ajuste de um modelo pré-treinado para uma tarefa específica, onde o modelo é treinado novamente com um conjunto de dados menor e mais especializado para adaptar-se à tarefa em questão [rag-knowledge-intensive-nlp.pdf:p2]. Isso significa que o modelo é ajustado para aprender padrões e relações específicas da tarefa, mas ainda depende da sua capacidade de armazenar conhecimento em sua arquitetura paramétrica.

Já o RAG é uma abordagem que combina um modelo paramétrico com um mecanismo de recuperação de informações, permitindo que o modelo acesse e utilize conhecimento externo armazenado em uma base de dados ou índice de recuperação [rag-knowledge-intensive-nlp.pdf:p2]. Isso permite que o modelo gere respostas mais específicas e factualmente corretas, pois 

In [10]:
QUERIES = [
    "Qual e a diferenca entre fine-tuning e RAG?",
    "O que e chunking e por que importa?",
    "Que metricas avaliam qualidade de retrieval?",
    "Como reduzir alucinacao em LLM?",
    "Qual o papel de embeddings em busca semantica?",
]

for q in QUERIES:
    out = rag_answer(q)
    print(f"\nQ: {q}")
    print(f"A: {out['answer']}")
    print(f"Fontes: {out['sources']}")


Q: Qual e a diferenca entre fine-tuning e RAG?
A: Nao encontrado no corpus.
Fontes: [('rag-knowledge-intensive-nlp.pdf', 2), ('rag-knowledge-intensive-nlp.pdf', 9), ('rag-knowledge-intensive-nlp.pdf', 5), ('rag-knowledge-intensive-nlp.pdf', 6), ('rag-knowledge-intensive-nlp.pdf', 1)]

Q: O que e chunking e por que importa?
A: Nao encontrado no corpus.
Fontes: [('lost-in-the-middle.pdf', 10), ('lost-in-the-middle.pdf', 2), ('lost-in-the-middle.pdf', 7), ('rag-knowledge-intensive-nlp.pdf', 19), ('attention-is-all-you-need.pdf', 3)]

Q: Que metricas avaliam qualidade de retrieval?
A: Nao encontrado no corpus.
Fontes: [('lost-in-the-middle.pdf', 8), ('rag-knowledge-intensive-nlp.pdf', 7), ('lost-in-the-middle.pdf', 7), ('lost-in-the-middle.pdf', 10), ('lost-in-the-middle.pdf', 6)]

Q: Como reduzir alucinacao em LLM?
A: Nao encontrado no corpus.
Fontes: [('rag-knowledge-intensive-nlp.pdf', 1), ('lost-in-the-middle.pdf', 9), ('attention-is-all-you-need.pdf', 9), ('lost-in-the-middle.pdf', 1

##célula de debug devido não ter encontrado o corpus

In [13]:
# Debug: ver o prompt completo e a resposta bruta do LLM para a 1ª query

q = "Qual e a diferenca entre fine-tuning e RAG?"
hits = retrieve(q, k=5)

print("=== CHUNKS RECUPERADOS ===")
for i, h in enumerate(hits):
    print(f"\n--- Chunk {i+1} [{h['source']}:p{h['page']}] ---")
    print(h['text'][:300])

context = "\n\n---\n\n".join(f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits)

full_prompt = RAG_PROMPT.format(context=context, question=q)

print("\n\n=== PROMPT COMPLETO (primeiros 1500 chars) ===")
print(full_prompt[:1500])
print(f"\n... (total: {len(full_prompt)} caracteres)")

print("\n\n=== RESPOSTA BRUTA DO LLM ===")
response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": full_prompt}],
    temperature=0.0,
)
print(response.choices[0].message.content)

=== CHUNKS RECUPERADOS ===

--- Chunk 1 [rag-knowledge-intensive-nlp.pdf:p2] ---
basis (where different documents are responsible for different tokens). Like T5 [51] or BART, RAG
can be ﬁne-tuned on any seq2seq task, whereby both the generator and retriever are jointly learned.
There has been extensive previous work proposing architectures to enrich systems with non-parametric
m

--- Chunk 2 [rag-knowledge-intensive-nlp.pdf:p9] ---
pieces of retrieved content, as well as learning latent retrieval, and retrieving evidence documents
rather than related training pairs. This said, RAG techniques may work well in these settings, and
could represent promising future work.
6 Discussion
In this work, we presented hybrid generation mod

--- Chunk 3 [rag-knowledge-intensive-nlp.pdf:p5] ---
4 Results
4.1 Open-domain Question Answering
Table 1 shows results for RAG along with state-of-the-art models. On all four open-domain QA
tasks, RAG sets a new state of the art (only on the T5-comparable split

## Verificação

- [ ] Ingestão: `docs` contém ≥10 páginas com `text` não-vazio
- [ ] Chunking: chunks têm tamanho médio próximo de 800 chars
- [ ] Indexing: `collection.count() == len(chunks)`
- [ ] Retrieval: top-5 retornados com `distance` crescente
- [ ] Generation: 5 respostas com citação de fonte; ≥3 com citação **correta** (verificável manualmente no PDF)

> **Próximo passo:** rodar `03-ragas-evaluation.ipynb` para avaliar **quantitativamente** a qualidade deste pipeline.